In [ ]:
import requests
from datetime import datetime, timedelta

def obtener_dato_hora_en_punto(id_estacion, id_sensor):
    # 1. Calculamos la ventana de tiempo (últimas 2 horas)
    ahora = datetime.utcnow()
    hace_dos_horas = ahora - timedelta(hours=2)
    
    # Formateamos las fechas según lo que suela pedir la API (ej. YYYY-MM-DDTHH:MM:SS)
    date_to = ahora.strftime('%Y-%m-%dT%H:%M:%S')
    date_from = hace_dos_horas.strftime('%Y-%m-%dT%H:%M:%S')
    page = 1
    
    # 2. Construimos la URL
    base_url = "URL_BASE_DE_LA_API_AQUI" # Hay que poner el dominio real
    endpoint = f"{base_url}/readings/station/{id_estacion}/sensor/{id_sensor}/from/{date_from}/to/{date_to}/{page}"
    
    try:
        # 3. Hacemos la petición
        respuesta = requests.get(endpoint)
        respuesta.raise_for_status()
        datos = respuesta.json()
        
        # 4. Filtramos para quedarnos solo con la hora en punto (:00:00)
        # (Dependerá de la estructura exacta del JSON devuelto)
        for lectura in datos: # Asumiendo que devuelve una lista
            if lectura['timestamp'].endswith(':00:00') or lectura['timestamp'].endswith(':00:00Z'):
                return lectura
                
    except Exception as e:
        print(f"Error al consultar la estación {id_estacion}: {e}")
        return None

# Ejemplo de uso:
# dato_limpio = obtener_dato_hora_en_punto(id_estacion=51, id_sensor=102)

In [ ]:
import pandas as pd

BASE_URL = "https://datos.tenerife.es/api/meteo/latest"

def mapear_estaciones():
    print("Obteniendo lista de estaciones...")
    url = f"{BASE_URL}/stations"
    
    try:
        respuesta = requests.get(url)
        respuesta.raise_for_status()
        estaciones_json = respuesta.json()
        
        # Convertimos a DataFrame para verlo bonito
        df_estaciones = pd.DataFrame(estaciones_json)
        print("Estaciones obtenidas con éxito.")
        return df_estaciones
        
    except requests.exceptions.RequestException as e:
        print(f"Error al conectar con /stations: {e}")
        return None

def mapear_medidas():
    print("Obteniendo lista de medidas/instrumentos...")
    url = f"{BASE_URL}/measures" # o /measures/instruments según lo que documente el manual
    
    try:
        respuesta = requests.get(url)
        respuesta.raise_for_status()
        medidas_json = respuesta.json()
        
        df_medidas = pd.DataFrame(medidas_json)
        print("✅ Medidas obtenidas con éxito.")
        return df_medidas
        
    except requests.exceptions.RequestException as e:
        print(f"Error al conectar con /measures: {e}")
        return None

# ejecucion
if __name__ == "__main__":
    df_est = mapear_estaciones()
    df_med = mapear_medidas()
    
    # Si la petición fue exitosa, muestra las primeras filas
    if df_est is not None:
        print("\n--- MUESTRA DE ESTACIONES ---")
        print(df_est.head(50)) # Aquí se verá la columna del ID que necesitamos
        
    if df_med is not None:
        print("\n--- MUESTRA DE MEDIDAS ---")
        print(df_med.head(7)) # Aquí se verá los IDs de los sensores (temperatura, viento, etc.)

Obteniendo lista de estaciones...
✅ Estaciones obtenidas con éxito.
Obteniendo lista de medidas/instrumentos...
✅ Medidas obtenidas con éxito.

--- MUESTRA DE ESTACIONES ---
                                             stations
0   {'id_weatherstation': 1, 'name': 'ARICO_01', '...
1   {'id_weatherstation': 2, 'name': 'GALLETAS', '...
2   {'id_weatherstation': 5, 'name': 'GUIAIS01', '...
3   {'id_weatherstation': 7, 'name': 'OROTAV01', '...
4   {'id_weatherstation': 9, 'name': 'HELECHO1', '...
5   {'id_weatherstation': 10, 'name': 'RAVELO01', ...
6   {'id_weatherstation': 11, 'name': 'TEJINA01', ...
7   {'id_weatherstation': 12, 'name': 'GUAN1HA1', ...
8   {'id_weatherstation': 13, 'name': 'VILAFLOR', ...
9   {'id_weatherstation': 54, 'name': 'TOPONEGR', ...
10  {'id_weatherstation': 55, 'name': 'ABONACOP', ...
11  {'id_weatherstation': 56, 'name': 'TEGUESTE', ...
12  {'id_weatherstation': 57, 'name': 'LLANITOP', ...
13  {'id_weatherstation': 58, 'name': 'RATINO 01',...
14  {'id_weather

In [9]:
url = "https://datos.tenerife.es/api/meteo/latest/stations"
respuesta = requests.get(url)

if respuesta.status_code == 200:
    datos = respuesta.json()
    # Aplanamos el JSON
    df_estaciones_agro = pd.json_normalize(datos['stations'])
    
    # Nos quedamos con los datos geográficos de interés
    df_mapa = df_estaciones_agro[['id_weatherstation', 'name', 'municipality_name', 'latitude', 'longitude', 'altitude']]
    
    # Lo guardamos en un CSV para que puedas revisarlo o meterlo en tu modelo
    df_mapa.to_csv("estaciones_agrocabildo_coordenadas.csv", index=False)
    
    print(f"¡Éxito! Se han guardado las coordenadas de {len(df_mapa)} estaciones.")
    print(df_mapa.head(50))
else:
    print("Error al conectar con la API")

¡Éxito! Se han guardado las coordenadas de 67 estaciones.
    id_weatherstation          name           municipality_name  \
0                   1      ARICO_01                       Arico   
1                   2      GALLETAS                       Arona   
2                   5      GUIAIS01               Guía de Isora   
3                   7      OROTAV01                  La Orotava   
4                   9      HELECHO1                       Arico   
5                  10      RAVELO01                   El Sauzal   
6                  11      TEJINA01  San Cristóbal de La Laguna   
7                  12      GUAN1HA1           Icod de los Vinos   
8                  13      VILAFLOR                    Vilaflor   
9                  54      TOPONEGR                      Güímar   
10                 55      ABONACOP                       Arico   
11                 56      TEGUESTE                    Tegueste   
12                 57      LLANITOP           Icod de los Vinos   
13  

In [10]:
import folium

# 1. Cargar las coordenadas que guardamos en el paso anterior
# Si tienes tu DataFrame cargado en memoria (df_mapa), puedes omitir esta línea
df_estaciones = pd.read_csv("estaciones_agrocabildo_coordenadas.csv")

# 2. Crear el mapa base centrado en la isla de Tenerife
# Coordenadas centrales aproximadas de Tenerife y un zoom inicial adecuado
mapa_tenerife = folium.Map(location=[28.291565, -16.629130], zoom_start=10)

# 3. Recorrer el DataFrame para añadir cada estación al mapa
for indice, fila in df_estaciones.iterrows():
    # Extraer los datos de la fila
    lat = fila['latitude']
    lon = fila['longitude']
    nombre = fila['name']
    municipio = fila['municipality_name']
    altitud = fila['altitude']
    id_estacion = fila['id_weatherstation']
    
    # Crear el texto que aparecerá al hacer clic en el marcador (Popup)
    # Usamos HTML básico para que se vea ordenado y claro
    info_popup = f"""
    <b>Estación:</b> {nombre} (ID: {id_estacion})<br>
    <b>Municipio:</b> {municipio}<br>
    <b>Altitud:</b> {altitud} m
    """
    
    # Añadir el marcador al mapa
    folium.Marker(
        location=[lat, lon],
        popup=folium.Popup(info_popup, max_width=300),
        tooltip=nombre, # Esto muestra el nombre al pasar el ratón por encima
        icon=folium.Icon(color='blue', icon='cloud', prefix='fa') # Icono de nube (FontAwesome)
    ).add_to(mapa_tenerife)

# 4. Guardar el mapa en un archivo HTML interactivo
archivo_salida = "mapa_estaciones_agrocabildo.html"
mapa_tenerife.save(archivo_salida)

print(f"¡Mapa generado con éxito! Abre el archivo '{archivo_salida}' en tu navegador web.")

¡Mapa generado con éxito! Abre el archivo 'mapa_estaciones_agrocabildo.html' en tu navegador web.
